# Homework 3

Lead scoring, 2026 release. Target is `converted`.


In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, mutual_info_score

url = 'https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv'
df = pd.read_csv(url)
df.head()


,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [2]:
df.isna().sum()


lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [3]:
cat = ['lead_source', 'industry', 'employment_status', 'location']
num = ['annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score']

for c in cat:
    df[c] = df[c].fillna('NA')
for c in num:
    df[c] = df[c].fillna(0.0)

df.isna().sum().sum()


np.int64(0)

## Q1 - mode for industry


In [4]:
df['industry'].value_counts()


industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240
Name: count, dtype: int64

**Q1: technology** - 1173 rows, ahead of retail at 925.

Worth noting `NA` is not the answer even though 240 rows were missing. Filling them all
with one label still leaves technology on top.


## Q2 - biggest correlation


In [5]:
df[num].corr().round(4)


,annual_income,number_of_courses_viewed,interaction_count,lead_score
annual_income,1.0000,0.1613,0.1228,0.2295
number_of_courses_viewed,0.1613,1.0000,0.7216,0.7572
interaction_count,0.1228,0.7216,1.0000,0.9157
lead_score,0.2295,0.7572,0.9157,1.0000


In [6]:
corr = df[num].corr()
pairs = [('interaction_count', 'lead_score'),
         ('number_of_courses_viewed', 'lead_score'),
         ('number_of_courses_viewed', 'interaction_count'),
         ('annual_income', 'interaction_count')]

pd.Series({f'{a} / {b}': corr.loc[a, b] for a, b in pairs}).sort_values(ascending=False)


interaction_count / lead_score                  0.915746
number_of_courses_viewed / lead_score           0.757204
number_of_courses_viewed / interaction_count    0.721609
annual_income / interaction_count               0.122842
dtype: float64

**Q2: `interaction_count` and `lead_score`**, at 0.916.

All three of the course-viewing and interaction pairs are strongly correlated with each
other. `annual_income` barely moves with anything, 0.12 at most.


## Split


In [7]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train['converted'].values
y_val = df_val['converted'].values
y_test = df_test['converted'].values

del df_train['converted']
del df_val['converted']
del df_test['converted']

len(df_train), len(df_val), len(df_test)


(3000, 1000, 1000)

## Q3 - biggest mutual information


In [8]:
mi = {c: mutual_info_score(df_train[c], y_train) for c in cat}
pd.Series(mi).sort_values(ascending=False).round(2)


lead_source          0.03
employment_status    0.02
industry             0.00
location             0.00
dtype: float64

In [9]:
pd.Series(mi).sort_values(ascending=False)


lead_source          0.034346
employment_status    0.018966
industry             0.001995
location             0.001043
dtype: float64

**Q3: `lead_source`**, 0.03.

`employment_status` comes second at 0.02. `industry` and `location` round to 0.00, so
whatever shows up in Q1 about industry is not telling us much about conversion.


## Q4 - logistic regression accuracy


In [10]:
features = cat + num


def train_and_score(columns, C=1.0):
    dv = DictVectorizer(sparse=False)
    X_train = dv.fit_transform(df_train[columns].to_dict(orient='records'))
    X_val = dv.transform(df_val[columns].to_dict(orient='records'))

    model = LogisticRegression(solver='liblinear', C=C, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)

    return accuracy_score(y_val, model.predict(X_val))


accuracy = train_and_score(features)
accuracy, round(accuracy, 2)


(0.645, 0.65)

**Q4: 0.65** (0.645 before rounding).


## Q5 - feature elimination


In [11]:
rows = []
for f in features:
    without = train_and_score([c for c in features if c != f])
    rows.append({'excluded': f, 'accuracy': without, 'difference': accuracy - without})

pd.DataFrame(rows).set_index('excluded').sort_values('difference')


,accuracy,difference
excluded,,
annual_income,0.724,-0.079
industry,0.645,0.000
location,0.645,0.000
employment_status,0.645,0.000
lead_score,0.644,0.001
number_of_courses_viewed,0.643,0.002
lead_source,0.642,0.003
interaction_count,0.601,0.044


**Q5: `number_of_courses_viewed`**, difference of +0.002. `lead_source` is +0.003 and
`interaction_count` +0.044.

The margin between the first two is 0.001, which is a single row out of 1000 in the
validation set. I re-ran the whole thing with `pd.get_dummies` instead of
`DictVectorizer` to make sure the ordering was not an artefact of the encoder, and got
the same three numbers.

The more interesting line in that table is `annual_income`: dropping it takes accuracy
from 0.645 to 0.724. It is actively hurting the model. It is not one of the options, but
it is the feature I would actually remove.


## Q6 - regularisation


In [12]:
scores = {}
for C in [0.000001, 0.00001, 0.0001, 0.001]:
    scores[C] = round(train_and_score(features, C=C), 3)

pd.Series(scores)


0.000001    0.598
0.000010    0.598
0.000100    0.613
0.001000    0.645
dtype: float64

**Q6: 0.001**, accuracy 0.645.

Accuracy climbs the whole way across the range, so the best C is the largest one on the
list. The two smallest values both sit at 0.598, which is roughly the share of the
majority class - the model is regularised so hard it stops using the features at all.
